# The challenge data exploration

This notebook is for **competitors** who want to understand:

1. **Single-cell RNA-seq** in plain language — why we work at cell resolution (with a pointer to the [10x Genomics guide](https://www.10xgenomics.com/what-is-single-cell-rna-seq)).
2. **AnnData** and the **`.h5ad`** file format used across single-cell tooling (Scanpy, scVI, etc.).
3. Practical setup and lightweight inspection (**A1**, **A2**).
4. **Pseudobulk** concept and method used in this repository.
5. Donor-aggregated pseudobulk exploration (**A3**).
6. **Genotyping files in this challenge** — VCF, GT TSV, and genotype PCs.

**Official AnnData documentation:** [anndata.readthedocs.io](https://anndata.readthedocs.io/en/latest/index.html) — the format sits between pandas and tensor-style arrays and is central to the [scverse](https://scverse.org/) ecosystem.

**Requirements:** `scanpy`, `pandas`, `numpy` (same environment as the baseline model tutorial).

**New to biology?** Read **`00_biology_genome_and_ngs_primer.ipynb`** first (genome → RNA-seq → `.h5ad`).

## Single-cell RNA sequencing (scRNA-seq)

The competition data come from **single-cell** experiments: each observation in the cell-level `.h5ad` is one cell (or one cell type within one donor after aggregation). The following overview follows the beginner-oriented framing in the **10x Genomics** learning hub — [What is single cell RNA sequencing?](https://www.10xgenomics.com/what-is-single-cell-rna-seq).

**What it measures.** Single-cell RNA sequencing measures **gene expression in individual cells** (typically thousands of genes per cell). That lets you recover **which genes are expressed in which cells**, instead of averaging over a whole tissue or blood sample.

**Why not only bulk RNA-seq?** In **bulk** RNA-seq, expression is averaged across many cells in one tube. That can hide rare populations and mixed cell states. For example, in a tumour sample, a small subset of treatment-resistant cells might be invisible in bulk data but detectable when each cell is measured separately. scRNA-seq is often used to map **heterogeneity**, **rare cell types**, and **dynamic states** that bulk methods miss.

**What researchers use it for** (examples from the same resource): building atlases of healthy vs diseased tissue, tracking developmental or immune dynamics, studying **cell–cell** communication in niches such as tumours, and defining new cell types or states with genome-wide readouts rather than only a fixed marker panel.

**Trade-offs (high level).** Single-cell experiments usually need **more depth and specialised analysis** than bulk, and workflows can be more involved — but the field standardises on shared objects (here: **AnnData** / `.h5ad`) and tools such as Scanpy so downstream steps are reproducible.

*Source: 10x Genomics, “A beginner's guide to building confidence in single cell RNA-seq” — [link](https://www.10xgenomics.com/what-is-single-cell-rna-seq).*


## Part A — What is AnnData?

We first build a compact understanding of the `AnnData` object and the challenge `.h5ad` structure, then move to practical exploration steps (**A1**, **A2**), pseudobulk construction, and finally genotyping files.


## What is AnnData?

An **`AnnData`** object (`anndata.AnnData`) bundles:

| Slot | Role | Typical content |
|------|------|-----------------|
| **`X`** | Main data matrix | `n_obs × n_vars` (cells × genes), often sparse counts |
| **`obs`** | Row annotations | cell IDs, QC, `donor_id`, `celltype`, `age`, … |
| **`var`** | Column annotations | gene symbols / Ensembl IDs, optional flags |
| **`layers`** | Extra matrices | raw counts, normalized data, imputed |
| **`obsm` / `varm`** | Multi-column embeddings | PCA, UMAP coordinates |
| **`uns`** | Unstructured dict | pipeline parameters, colors, … |

**Naming:** `obs` = observations (here: **cells**, or later **donor×celltype** rows). `var` = variables (here: **genes**).

**.h5ad** is the HDF5-based on-disk format for AnnData (`read_h5ad` / `write_h5ad`). See the AnnData docs for [on-disk layout and interoperability](https://anndata.readthedocs.io/en/latest/index.html).

**Teaching point:** Inspect `adata.shape`, `adata.obs.head()`, and `adata.var_names` first whenever you open a new file.

### Visual schema

![AnnData schematic: matrix X with obs (rows) and var (columns) and linked annotation slots](../images/anndata_schema_image.svg)

### A1 — Project path

Run from the repository root or from `notebooks/`.

In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "models" / "train_age_model.py").exists():
    PROJ_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "models" / "train_age_model.py").exists():
    PROJ_ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError(
        "Could not find models/train_age_model.py. Open Jupyter from the repo root or from notebooks/."
    )

# Competition data live under `data/` (bind or copy from shared scratch — see README).
DATA_DIR = PROJ_ROOT / "data"
RESULTS_DIR = PROJ_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJ_ROOT)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)

### A2 — Explore competition **cell-level** `.h5ad` (lightweight peek)

Full objects (~1M+ cells) are **large in RAM**. Two strategies:

1. **`backed='r'`** — keep `X` on disk while you inspect `obs` / `var` (see [AnnData backed mode](https://anndata.readthedocs.io/en/latest/generated/anndata.AnnData.html)).
2. **Pseudobulk outputs** (next sections) — much smaller; use those for quick plots.

Files are read from **`data/`** (e.g. `data/scRNA-seq_raw/train.h5ad`). Adjust `CELL_LEVEL` only if you use a custom layout.

In [ ]:
import scanpy as sc

CELL_LEVEL = DATA_DIR / "scRNA-seq_raw" / "train.h5ad"

if not CELL_LEVEL.exists():
    print("No cell-level file at:", CELL_LEVEL)
    print("Copy or bind competition data into data/ (see README), or skip to pseudobulk sections.")
else:
    ad_cell = sc.read_h5ad(CELL_LEVEL, backed="r")
    try:
        print("Shape (obs × var):", ad_cell.n_obs, "×", ad_cell.n_vars)
        print("obs columns (sample):", list(ad_cell.obs.columns[:12]), "...")
        print(ad_cell.obs[["donor_id", "celltype", "age", "_split"]].head(6))
    finally:
        if getattr(ad_cell, "isbacked", False) and ad_cell.file is not None:
            ad_cell.file.close()

**Competition columns (cell-level `obs`):** `donor_id`, `celltype` (5 coarse groups), `age` (missing on test cells in some builds), `_split` (`train` / `val` / `test`), plus QC fields — see the main `README.md`.

Each **row** is one cell; `X[i, g]` is the expression of gene `g` in cell `i`.

## Part B — Pseudobulk explain and method

For competitors, the key file is already prepared and shipped:

- `data/scRNA-seq_pseudobulk/train_pseudobulk_donor_aggregated_public.h5ad`

This donor-level matrix is what the baseline model uses as input features.

### How it is constructed (conceptual)

From cell-level `AnnData`, for each cell type group we compute:

$$ \text{pseudobulk}_{d,c,g} = \sum_{\text{cells } i \text{ with donor } d,\, \text{type } c} X_{i,g} $$

Then features are reshaped to donor-level wide format (one row per donor) with names like `GENE__CD4_T`, `GENE__CD8_T`, etc.

**Why pseudobulk?** It collapses millions of cells into **interpretable donor-level features**, reduces single-cell noise, and matches donor-level prediction targets (here: age).

**Caveat:** Summing raw counts assumes within-cell-type comparability; depth differences across donors can still matter (the baseline applies `log1p` before Random Forest).

### A3 — Explore train **donor-aggregated** pseudobulk `.h5ad`

Use the train split donor-level matrix directly:

- `data/scRNA-seq_pseudobulk/train_pseudobulk_donor_aggregated_public.h5ad`

This matrix has one row per donor and feature columns like `ENSG...__CD4_T`, `ENSG...__CD8_T`, etc. (5 cell-type slots per gene).

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display

PB_TRAIN = DATA_DIR / "scRNA-seq_pseudobulk" / "train_pseudobulk_donor_aggregated_public.h5ad"

if not PB_TRAIN.exists():
    print("No train donor-aggregated pseudobulk file at:", PB_TRAIN)
    print("Copy or bind competition data into data/ (see data/README.md).")
else:
    ad_pb = sc.read_h5ad(PB_TRAIN)
    print("Shape:", ad_pb.n_obs, "donors ×", ad_pb.n_vars, "features")
    print("obs columns:", list(ad_pb.obs.columns))

    obs_cols = [c for c in ("donor_id", "age", "_split") if c in ad_pb.obs.columns]
    display(ad_pb.obs[obs_cols].head(10))

    # Features live in X/var as gene__celltype columns
    g0 = ad_pb.var_names[0]
    print(f"Example feature {g0!r} — first donors:")
    Xcol = ad_pb[:10, g0].X
    if hasattr(Xcol, "toarray"):
        Xcol = Xcol.toarray()
    values = np.asarray(Xcol).ravel()

    demo = ad_pb.obs[obs_cols].head(10).copy() if obs_cols else pd.DataFrame(index=ad_pb.obs.index[:10])
    demo[g0] = values
    display(demo)

### A4 — **Donor-aggregated** matrix (what the baseline model reads)

`scRNA-seq_pseudobulk/train_pseudobulk_donor_aggregated_public.h5ad` (train split) has **one row per donor** and **one column per (gene, cell-type slot)** with names like `ENSG...__CD4_T`.

So each gene contributes **5 numbers** per donor (CD4 T, CD8 T, NK, B cells, monocytes). Total columns ≈ `5 × n_genes`. Use `val_` / `test_` files for those splits.

In [ ]:
PB_AGG = DATA_DIR / "scRNA-seq_pseudobulk" / "train_pseudobulk_donor_aggregated_public.h5ad"

if not PB_AGG.exists():
    print("No donor-aggregated file at:", PB_AGG)
else:
    ad_agg = sc.read_h5ad(PB_AGG)
    print("Shape:", ad_agg.n_obs, "donors ×", ad_agg.n_vars, "features")
    print("Example var_names:", list(ad_agg.var_names[:6]))
    display(ad_agg.obs[["donor_id", "age", "_split"]].head())

## Genotyping data in the challenge

The folder `data/genotypes` contains three useful file groups:

- **VCF**: raw variant-level genotype calls (`0/0`, `0/1`, `1/1`) with metadata columns.
- **GT TSV**: model-friendly donor-by-variant matrix for quick ML exploration.
- **PC files**: low-dimensional genotype principal components (`pca_train.tsv`, `pca_val.tsv`, `pca_test.tsv`) often used as covariates.

![VCF files overview](../images/VCF_files.jpg)

Use `donor_id` to align these with RNA/pseudobulk donor-level features.

### Explore VCF quickly

In [ ]:
import gzip
from pathlib import Path

GENO_DIR = DATA_DIR / "genotypes"
vcf_candidates = sorted(GENO_DIR.glob("*.vcf")) + sorted(GENO_DIR.glob("*.vcf.gz"))

if not vcf_candidates:
    print("No .vcf/.vcf.gz found in", GENO_DIR)
else:
    vcf_path = vcf_candidates[0]
    print("VCF file:", vcf_path)

    opener = gzip.open if vcf_path.suffix == ".gz" else open
    with opener(vcf_path, "rt") as f:
        header = None
        first_variant = None
        for line in f:
            if line.startswith("##"):
                continue
            if line.startswith("#CHROM"):
                header = line.rstrip("\n").split("\t")
                continue
            first_variant = line.rstrip("\n").split("\t")
            break

    if header is None:
        print("Could not find VCF header line (#CHROM).")
    else:
        print("Header columns (first 10):")
        print(header[:10])

    if first_variant is not None:
        print("\nFirst variant row (first 10 columns):")
        print(first_variant[:10])
    else:
        print("No variant rows found in file.")

### Explore GT TSV quickly

In [ ]:
import pandas as pd

gt_candidates = sorted(GENO_DIR.glob("*gt*.tsv")) + sorted(GENO_DIR.glob("*.gt.tsv"))
if not gt_candidates:
    gt_candidates = [p for p in sorted(GENO_DIR.glob("*.tsv")) if "pca" not in p.name.lower()]

if not gt_candidates:
    print("No GT TSV found in", GENO_DIR)
else:
    gt_path = gt_candidates[0]
    print("GT TSV file:", gt_path)
    gt_df = pd.read_csv(gt_path, sep="\t", nrows=5)
    print("Columns (first 12):", list(gt_df.columns[:12]))
    display(gt_df.iloc[:, : min(12, gt_df.shape[1])])

### Explore PCA files quickly

In [ ]:
import pandas as pd

pca_paths = sorted(GENO_DIR.glob("pca_*.tsv"))
if not pca_paths:
    pca_paths = [p for p in sorted(GENO_DIR.glob("*.tsv")) if "pca" in p.name.lower()]

if not pca_paths:
    print("No PCA TSV files found in", GENO_DIR)
else:
    for p in pca_paths:
        df = pd.read_csv(p, sep="\t", nrows=5)
        print("\n", p.name, "shape(sample):", df.shape)
        print("columns:", list(df.columns))
        display(df)